In [ ]:
import itertools
import json
import time
from dataclasses import asdict, replace
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from uavloc.datasets import dataset_ref
from uavloc.filtering.empty_view import EmptyViewModel
from uavloc.plotting.flight import plot_end_error_cdf
from uavloc.simulation.arms import FilterParams, HeadSpec, build_correlator, estimator_factory
from uavloc.simulation.sequence import (all_fixes, flight_metrics, load_flight_set, per_flight_table, run_flights,
                                        village_bootstrap)
from uavloc.track import experiment_dir, load_runs, log_run, set_seed

# Params

In [ ]:
EXP = "032-empty-view-likelihood"
SEED = 42
DEVICE = "cpu"                    # CPU-only process: MPS corrupts CPU conv2d in the same process (bug-121)
ROUTE_SET = "sim-flights"
INITIAL_ERROR_M = {"normal": 0.0, "cold": 40.0}
SIGMA0_M = {"normal": 1.0, "cold": 25.0}
CORRELATOR = {
    "method": "segmentation", "seg_architecture": "lraspp",
    "seg_checkpoint_drone": "pocs/mobilenet-lraspp-speed/model-lraspp-bn-satellite.pt",
    "seg_checkpoint_sat": "pocs/mobilenet-lraspp-speed/model-lraspp-bn-satellite.pt",
    "seg_channels": "all", "drone_seg_gsd_m": 0.385, "match_gsd_m": 0.385, "matching": "weighted-ncc",
    "class_weights": [0.0, 1.0, 1.0, 0.5, 0.0, 0.5], "window": "rect", "search_scale": 1.7, "fov_w_deg": 60.0,
    "sat_feature_crop_pct": 0.03, "drone_feature_crop_pct": 0.0,
}
N_PARTICLES = 2000
TEMPERATURE = 0.5                 # 027.12 / 027.4, tuned in 027
Q_POS_M_PER_SQRT_S = 2.0
WINDOW_CAP_HALF_M = 100.0
WINDOW_MARGIN_M = 2.0
HEAD_RUN = "023.3"
HEAD_EXP = "023-learned-correlation-head"
HEAD_ENCODER_CHANNELS = [32, 32, 16]
HEAD_REFINE_CHANNELS = 16
HEAD_CLEAN_VILLAGES = ["bushcha-50p2422-25p8305", "ivannie-50p4543-25p7398", "maidan-50p2512-26p0155",
                       "maksymivka-48p8871-23p9317", "oblaznytsia-49p2703-24p1288", "pidhaichyky-48p5627-25p1788",
                       "pidhorodnie-49p5387-25p5199", "sadzhavka-48p5678-24p7897", "strilche-48p6821-25p5613",
                       "vesnianka-49p7175-27p3183", "zlynets-50p3697-25p7999"]   # villages the 023 head never trained on
MODEL_PATH = "artifacts/empty_view.json"
TUNING_DIR = "artifacts/tuning"
SCENARIO = "normal"
SPLIT = "tuning"
LIKELIHOOD = "head"
BETAS = [0.0, 0.5, 1.0, 2.0]
SHARD = 0
N_SHARDS = 1

In [ ]:
set_seed(SEED)

# Data

In [ ]:
set_config, specs = load_flight_set(ROUTE_SET, split=SPLIT)
len(specs)

# Model

In [ ]:
model_record = json.loads(Path(MODEL_PATH).read_text())
empty_view = EmptyViewModel(**model_record["model"])
empty_view

In [ ]:
def filter_params(empty_view: EmptyViewModel | None, beta: float) -> FilterParams:
    return FilterParams(TEMPERATURE, Q_POS_M_PER_SQRT_S, N_PARTICLES, 0, 0, 0.0, 0.0, 0.0, 0.0, WINDOW_CAP_HALF_M,
                        WINDOW_MARGIN_M, empty_view, beta)

In [ ]:
def head_spec() -> HeadSpec:
    return HeadSpec(experiment_dir(HEAD_EXP) / "artifacts" / f"head-{HEAD_RUN}.pt", tuple(HEAD_ENCODER_CHANNELS),
                    HEAD_REFINE_CHANNELS)

In [ ]:
def correlator_for(map_raster):
    return build_correlator(map_raster, CORRELATOR, DEVICE, head_spec() if LIKELIHOOD == "head" else None)

In [ ]:
factories = {f"particles|{beta}": estimator_factory("particles", filter_params(empty_view, beta))
             for beta in BETAS[SHARD::N_SHARDS]}
list(factories)

# Run

In [ ]:
outcomes = run_flights(set_config, specs, INITIAL_ERROR_M[SCENARIO], SIGMA0_M[SCENARIO], correlator_for, factories)

In [ ]:
summary = pd.DataFrame([{"beta": float(name.split("|")[1]), **flight_metrics(found)} for name, found in outcomes.items()])
summary.sort_values("mean_end_error_m")

In [ ]:
out = Path(TUNING_DIR)
out.mkdir(parents=True, exist_ok=True)
summary.to_csv(out / f"shard-{SHARD}.csv", index=False)
pd.concat([per_flight_table(found).assign(combo=name) for name, found in outcomes.items()]).to_csv(
    out / f"shard-{SHARD}-flights.csv", index=False)